# BioLangFusion construction and reference checks

The [dataset audit](blf_datasets.ipynb) identifies four available benchmarks, an inferred length filter and split limitations. [Token extraction](blf_token_embeddings.ipynb) prepares frozen NT v2, RNA-FM and ESM-2 features. This notebook assembles the codon grid, fusion heads and TextCNN predictor, then asks how the implemented comparison relates to BioLangFusion's published fungal-expression results.

The investigation separates three questions: does the code satisfy its shape and masking contracts, how do its fitted scores compare with the paper, and which differences persist when the convolution setting or training seed changes? Matching a published score is useful evidence, but does not establish identical data, preprocessing or implementation. This is baseline preparation under [Gantt/Details task 17](../../docs/Research_Project_Plan_Gantt.xlsx), with explicit reconstruction choices.

The model code is shared so later comparisons use the same implementations:

| File | Role |
| --- | --- |
| [`codon_grid.py`](../shared_code/fusion/codon_grid.py) | Places the native token streams on a common triplet grid |
| [`heads.py`](../shared_code/fusion/heads.py) | Concatenation, gated MIL and cross-attention |
| [`textcnn.py`](../shared_code/fusion/textcnn.py) | Convolutional prediction head |
| [`blf.py`](../shared_code/fusion/blf.py) | Cached streams, batching and assembled model |
| [`train.py`](../shared_code/fusion/train.py) | Training, validation-based stopping and scoring |

The [synthetic model tests](../../tests/test_fusion_model.py) check implementation properties without research caches or checkpoints. **This notebook itself is executable research work:** Run All opens caches, performs timing and a fresh short fit, requests training sweeps, and recalculates comparisons. Saved outputs can be read without running it.


---

## Setup

The cells below separate imports, project-module paths and experiment settings. Use the [notebook environment](../../README.md#notebook-dependencies) and the caches from step 2.

`CACHE_ROOT` and `RESULTS_DIR` retain paths relative to the kernel's working directory. Start from `Code/stage3/` to use the intended locations. The initial diagnostic prints PyTorch's version and Apple availability; the later input-loading cell chooses the device, including NVIDIA CUDA, and reports the actual selection.


In [ ]:
# Standard-library environment, paths and timing.
import hashlib
import json
import os
import sys
import time
from pathlib import Path

# Enable supported CPU fallbacks for Apple MPS before importing PyTorch.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Arrays, reference tables and result summaries.
import numpy as np
import pandas as pd

# Model construction, gradients and hardware availability.
import torch


In [1]:
# Find Code/shared_code while preserving the existing repository-root fallback.
CODE_DIR = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "shared_code").is_dir()
    ),
    Path.cwd() / "Code",
)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from shared_code.fusion import blf, codon_grid, heads, run_identity, textcnn, train
from shared_code.fusion import swap_results, token_embeddings

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("torch", torch.__version__, "| Apple GPU available:", torch.backends.mps.is_available())


torch 2.12.1 | Apple GPU available: True


In [ ]:
# Fungal expression is the retained build comparison; other tasks need their own protocol.
BENCHMARK = "fungal_expression"
CACHE_ROOT = Path("embeddings")  # relative to the kernel working directory
RESULTS_DIR = Path("results") / BENCHMARK

CHANNELS = 100  # shared TextCNN width in the main comparison
BATCH_SIZE = 32
MAX_EPOCHS = 500
LAMBDAS = (0.01, 0.5, 1.0)  # positive entropy weights selected on validation
SELECTION_SEED = 0  # fixed seed-0 validation policy required by the shared selector
CLAIM_LAMBDA = 0.01  # recorded Equation 1 choice, held fixed across the repeat cohort
SEEDS = (0, 1, 2)  # later Equation 1 repeats
RUN_WIDE_HEAD = False  # optional 1,280-channel MIL fit near the end

# This existing report qualifies only its exact historical prediction files.
FUNGAL_VERIFICATION = (
    CODE_DIR / "stage3/results/fungal_expression/records/"
    "equation-verification-20261020T034236791322Z.json"
)
FUNGAL_VERIFICATION_SHA256 = "e8a3484aab0e0549325dca779afa00249c6847cba62cad5f31e68be017ea3b8d"


---

## The target

The [BioLangFusion paper](../../Papers/core/BioLangFusion.pdf), Table 1, compares seven single-encoder references and three fusion heads across five benchmarks. Table 4 adds gated-head ablations. The following tables transcribe those reference values; they are not local measurements.

This notebook fits the original three single encoders and fusion heads, including MIL with and without entropy. It displays the shared-projection ablation for context but does not implement that additional ablation here. E. coli's published accuracy is expressed as a percentage; the retained fitting and reporting route below is the fungal regression comparison.


In [2]:
# BioLangFusion Table 1 in full, as printed. Spearman correlation everywhere except E. coli,
# which is accuracy on a 0 to 100 scale. The first four rows are encoders the paper compares
# against but does not fuse; rows five to seven are the three it fuses.
TABLE1 = pd.DataFrame(
    [
        ("Evo", "other", 0.653, 0.579, 42.556, 0.403, 0.360),
        ("SpliceBERT", "other", 0.802, 0.778, 48.455, 0.522, 0.718),
        ("ESM-2 (650M)", "other", 0.825, 0.734, 46.348, 0.536, 0.679),
        ("ESM-2 (3B)", "other", 0.772, 0.721, 46.208, 0.537, 0.700),
        ("ESM-2 (8M)", "single", 0.806, 0.695, 49.017, 0.539, 0.711),
        ("RNA-FM", "single", 0.841, 0.767, 52.949, 0.553, 0.743),
        ("Nucleotide Transformer", "single", 0.780, 0.804, 41.292, 0.530, 0.732),
        ("Concatenation", "fusion", 0.831, 0.805, 50.280, 0.539, 0.764),
        ("MIL + Entropy", "fusion", 0.864, 0.824, 52.107, 0.563, 0.760),
        ("Cross Attention", "fusion", 0.828, 0.812, 53.932, 0.550, 0.765),
    ],
    columns=[
        "encoding",
        "kind",
        "cov_vac_degradation",
        "fungal_expression",
        "ecoli_proteins",
        "mrna_stability",
        "ab1",
    ],
).set_index("encoding")

# Table 4, the two ablations of the gated head, on the same five benchmarks.
TABLE4 = pd.DataFrame(
    [
        ("MIL without entropy", 0.854, 0.826, 51.960, 0.556, 0.753),
        ("MIL with shared projection (+ entropy)", 0.859, 0.813, 50.140, 0.563, 0.755),
        ("MIL + entropy", 0.864, 0.824, 52.107, 0.563, 0.760),
    ],
    columns=[
        "ablation",
        "cov_vac_degradation",
        "fungal_expression",
        "ecoli_proteins",
        "mrna_stability",
        "ab1",
    ],
).set_index("ablation")

display(TABLE1)
display(TABLE4)


,kind,cov_vac_degradation,fungal_expression,ecoli_proteins,mrna_stability,ab1
encoding,,,,,,
Evo,other,0.653,0.579,42.556,0.403,0.360
SpliceBERT,other,0.802,0.778,48.455,0.522,0.718
ESM-2 (650M),other,0.825,0.734,46.348,0.536,0.679
ESM-2 (3B),other,0.772,0.721,46.208,0.537,0.700
ESM-2 (8M),single,0.806,0.695,49.017,0.539,0.711
RNA-FM,single,0.841,0.767,52.949,0.553,0.743
Nucleotide Transformer,single,0.780,0.804,41.292,0.530,0.732
Concatenation,fusion,0.831,0.805,50.280,0.539,0.764
MIL + Entropy,fusion,0.864,0.824,52.107,0.563,0.760


,cov_vac_degradation,fungal_expression,ecoli_proteins,mrna_stability,ab1
ablation,,,,,
MIL without entropy,0.854,0.826,51.960,0.556,0.753
MIL with shared projection (+ entropy),0.859,0.813,50.140,0.563,0.755
MIL + entropy,0.864,0.824,52.107,0.563,0.760


On fungal expression, NT has the highest single-encoder score in Table 1 at 0.804. It is also one of the three encoders used in the fusion. Concatenation scores 0.805 and gated MIL 0.824. These are the published differences that motivate checking whether fusion improves on the NT baseline in our implementation.

The notebook checks absolute differences and model ordering separately. A similar absolute score can conceal a different ordering, while an ordering from one fitted seed is not by itself evidence of a reliable difference.

Table 4 also motivates retaining the zero-entropy ablation: on fungal expression, MIL without entropy scores 0.826 versus 0.824 with it. That reported comparison does not support an entropy benefit on this benchmark. The local positive-weight sweep and zero-weight fit let us examine the distinction under our implementation.


## Why this checks against fungal expression

Fungal expression supplies 3,138 retained rows, a regression target and published references for the compared variants. The [dataset audit](blf_datasets.ipynb#Published-splits,-and-how-much-the-test-split-reuses) found no exact sequence shared across its retained partitions. This makes it a useful initial build comparison; it does not establish independence of related genes or species, nor prove the build correct if scores match. The [target guide](../../docs/methods/data-and-targets.md#fungal-expression) retains the unresolved source-scale details.

The other benchmarks ask different questions. Stability has exact train/test sequence overlap, so its published-split score mixes seen and unseen sequence strings. Repeated labels can reflect different experimental profiles, not automatically damaged measurements. That split can support a qualified comparison with the published protocol, while genuinely unseen-sequence evaluation remains separate.

CoV-Vac supplies short designed RNA fragments whose protein inputs depend on the computational translation rule from step 2. E. coli supplies expression classes and therefore a classification loss and accuracy metric. These are useful later comparisons, but changing the benchmark also changes the biological target and evaluation requirements.


## What the paper leaves open

The shared implementation records several choices needed to turn the paper into executable code. The checks below examine those choices' implemented properties; they do not prove complete experimental equivalence.

1. **Two DNA upsampling specifications.** Equation 1 gives kernel 2 and stride 2, producing `2n` positions for `n` input tokens. The appendix gives kernel 3, stride 2 and padding 2, producing `2n - 3`. The first sweep retains the appendix default; a later sweep compares Equation 1. Both use row-specific adjustment to the triplet frame. Exact factor-two correspondence applies to complete six-mers, not NT's single-base remainder tokens. See [the codon-grid implementation](../shared_code/fusion/codon_grid.py).

2. **TextCNN width.** The paper describes a shared fixed prediction head, but lists 1,280 channels for gated MIL and 100 for cross-attention. The main comparison holds the output-channel count at 100 across variants, controlling that particular capacity difference. This does not equalize every model's capacity or optimization. A separately enabled fit examines 1,280 channels.

3. **DNA feature width and projection.** The selected checkpoint supplies 512-wide DNA features, alongside RNA-FM's 640 and ESM-2's 320. The paper's stated 4,107 corresponds to the DNA vocabulary size rather than these hidden states; DNA is therefore not the widest stream. The [fusion implementation](../shared_code/fusion/heads.py) retains the projection and uses the paper's named common width of 600.

4. **Stopping and scheduling.** The shared trainer monitors validation score for both mechanisms and restores the best validation state before final test scoring. These are explicit implementation choices where the paper leaves the monitored quantity unspecified.

5. **Padding and masked means.** Gated summaries average real codon positions, and each DNA row is limited to its own convolution output before frame masking. Otherwise, a sequence's representation could depend on padding introduced by its batch mates. The following CPU-tensor checks cover representative dimensions, batch-padding behavior, gate normalization and gradient flow.

The structural cells run in order and accumulate one report. Later cases reuse tensors constructed in earlier cases; the seed remains fixed at the start so their random operations retain their original order.


In [ ]:
# Structural checks, on random tensors and on the CPU: no cache, no checkpoint, no training.
# These check implemented shapes, masks and gradients, not complete paper fidelity.
torch.manual_seed(0)
ENCODER_WIDTHS = {"dna": 512, "rna": 640, "prot": 320}  # the widths step 2's caches actually hold
structural_checks = []


def record_structural_check(head_name, passed, detail):
    structural_checks.append(
        {"check": head_name, "result": "ok" if passed else "PROBLEM", "detail": detail}
    )


In [ ]:
# 1. Every stream lands on the protein frame, under both published convolution settings.
for setting in codon_grid.TCONV_SETTINGS:
    grid = codon_grid.CodonGrid(ENCODER_WIDTHS["dna"], tconv=setting).eval()
    sequence_length = 999
    dna = torch.randn(2, sequence_length // 6 + sequence_length % 6, ENCODER_WIDTHS["dna"])
    rna = torch.randn(2, sequence_length, ENCODER_WIDTHS["rna"])
    prot = torch.randn(2, sequence_length // 3, ENCODER_WIDTHS["prot"])
    codons = torch.tensor([sequence_length // 3] * 2)
    dna_token_lengths = torch.tensor([sequence_length // 6 + sequence_length % 6] * 2)
    with torch.no_grad():
        streams, mask = grid(dna, rna, prot, codons, dna_token_lengths)
    shapes = {modality: stream.shape[1] for modality, stream in streams.items()}
    record_structural_check(
        f"codon frame, tconv={setting}",
        set(shapes.values()) == {sequence_length // 3},
        f"999 nt -> {shapes}, expected {sequence_length // 3} everywhere",
    )


In [ ]:
# 2. A row is resampled the same alone as it is beside a longer row.
grid = codon_grid.CodonGrid(ENCODER_WIDTHS["dna"]).eval()
short_length, long_length = 300, 999
dna = torch.zeros(2, long_length // 6 + long_length % 6, ENCODER_WIDTHS["dna"])
rna = torch.zeros(2, long_length, ENCODER_WIDTHS["rna"])
prot = torch.zeros(2, long_length // 3, ENCODER_WIDTHS["prot"])
for row, n in enumerate((short_length, long_length)):
    dna[row, : n // 6 + n % 6] = torch.randn(n // 6 + n % 6, ENCODER_WIDTHS["dna"])
    rna[row, :n] = torch.randn(n, ENCODER_WIDTHS["rna"])
    prot[row, : n // 3] = torch.randn(n // 3, ENCODER_WIDTHS["prot"])
codons = torch.tensor([short_length // 3, long_length // 3])
dna_token_lengths = torch.tensor(
    [short_length // 6 + short_length % 6, long_length // 6 + long_length % 6]
)
with torch.no_grad():
    batched, _ = grid(dna, rna, prot, codons, dna_token_lengths)
    alone, _ = grid(
        dna[:1, : short_length // 6 + short_length % 6],
        rna[:1, :short_length],
        prot[:1, : short_length // 3],
        codons[:1],
        dna_token_lengths[:1],
    )
batch_difference = max(
    float((batched[modality][:1, : short_length // 3] - alone[modality]).abs().max())
    for modality in alone
)
record_structural_check(
    "a row is unchanged by its batch mates",
    batch_difference == 0.0,
    f"largest difference {batch_difference:.3g}",
)


In [ ]:
# 3. The prediction head ignores padding.
head = textcnn.TextCNN(ENCODER_WIDTHS["rna"], 1, channels=16).eval()
real = torch.randn(1, 120, ENCODER_WIDTHS["rna"])
padded = torch.cat([real, torch.randn(1, 60, ENCODER_WIDTHS["rna"])], dim=1)
padded_mask = torch.zeros(1, 180, dtype=torch.bool)
padded_mask[:, :120] = True
with torch.no_grad():
    padding_difference = float(
        (head(real, torch.ones(1, 120, dtype=torch.bool)) - head(padded, padded_mask)).abs().max()
    )
record_structural_check(
    "the head ignores padding",
    padding_difference < 1e-5,
    f"largest difference {padding_difference:.3g}",
)


In [ ]:
# 4. The gated head gives one weight per modality per sequence, summing to one.
streams, mask = grid(dna, rna, prot, codons, dna_token_lengths)
_, aux = heads.GatedMILFusion(ENCODER_WIDTHS)(streams, mask)
modality_weights = aux["alpha"]
record_structural_check(
    "gated attention: one weight per modality per sequence",
    modality_weights.shape == (2, 3)
    and bool(torch.allclose(modality_weights.sum(1), torch.ones(2))),
    f"shape {tuple(modality_weights.shape)}, row sums {[round(float(v), 6) for v in modality_weights.detach().sum(1)]}",
)


In [3]:
# 5. Every parameter of every head receives a finite gradient.
for head_name, build_head in heads.FUSIONS.items():
    model = build_head(ENCODER_WIDTHS)
    fused, _ = model({modality: stream.detach() for modality, stream in streams.items()}, mask)
    fused.sum().backward()
    invalid_gradients = [
        n
        for n, parameter in model.named_parameters()
        if parameter.requires_grad
        and (parameter.grad is None or not torch.isfinite(parameter.grad).all())
    ]
    record_structural_check(
        f"gradients reach every parameter, {head_name}",
        not invalid_gradients,
        "all finite" if not invalid_gradients else f"missing: {invalid_gradients}",
    )

pd.DataFrame(structural_checks).set_index("check")


,result,detail
check,,
"codon frame, tconv=appendix",ok,"999 nt -> {'dna': 333, 'rna': 333, 'prot': 333..."
"codon frame, tconv=equation",ok,"999 nt -> {'dna': 333, 'rna': 333, 'prot': 333..."
a row is unchanged by its batch mates,ok,largest difference 0
the head ignores padding,ok,largest difference 0
gated attention: one weight per modality per sequence,ok,"shape (2, 3), row sums [1.0, 1.0]"
"gradients reach every parameter, concat",ok,all finite
"gradients reach every parameter, mil",ok,all finite
"gradients reach every parameter, cross",ok,all finite


---

## Open the cached benchmark


### Device, fit order and input streams

The settings are in Setup. The next cell preserves the original device preference—Apple MPS, then NVIDIA CUDA, then CPU—declares the fit order and opens the cached benchmark. It also creates the result directory, so this is an input/output step rather than configuration alone.

Completed results and checkpoints are handled by the later recorded-fit helper and shared trainer. Reuse depends on the available configuration and provenance checks; an incompatible or incomplete historical record is not permission to delete or replace it.


In [4]:
# Preserve this notebook's device preference: Apple MPS, then NVIDIA CUDA, then CPU.
DEVICE = (
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)

# Keep the original fit order; lambda zero is a separate ablation.
RUNS = (
    [
        ("unimodal:prot", {}),
        ("unimodal:dna", {}),
        ("unimodal:rna", {}),
        ("concat", {}),
    ]
    + [("mil", {"entropy_lambda": lam}) for lam in LAMBDAS]
    + [("mil", {"entropy_lambda": 0.0}), ("cross", {})]
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
STREAMS = blf.BenchmarkStreams.open(BENCHMARK, CACHE_ROOT)
print(
    f"{BENCHMARK}: {len(STREAMS.rows):,} rows, {STREAMS.task}, "
    f"widths {STREAMS.widths}, device {DEVICE}"
)
print("splits:", {s: len(STREAMS.split_rows(s)) for s in ("train", "val", "test")})
print(
    f"{len(RUNS)} runs:",
    ", ".join(v + (f" (lam {o['entropy_lambda']})" if o else "") for v, o in RUNS),
)


fungal_expression: 3,138 rows, regression, widths {'dna': 512, 'rna': 640, 'prot': 320}, device mps
splits: {'train': 2153, 'val': 555, 'test': 430}
9 runs: unimodal:prot, unimodal:dna, unimodal:rna, concat, mil (lam 0.01), mil (lam 0.5), mil (lam 1.0), mil (lam 0.0), cross


### Estimate training time

Opening `BenchmarkStreams` checks cache row counts and expected token lengths for the selected encoders. These structural checks do not establish identical sequence contents or biological correspondence; recorded-fit preparation separately checks the evidence required for new work.

The timing helpers construct each variant, perform one warm-up backward pass, then time three optimizer batches. The following cell extrapolates those batches to a training epoch and shows 100- and 250-epoch scenarios. Those durations are scenarios, not predictions of the stopping epoch.

Sequence lengths, padding, device and model architecture affect cost. These timings exclude model construction and validation; the GPU path has no explicit synchronization, so treat its wall-clock estimates cautiously. Failed measurements are printed separately, and the displayed total can omit their NaN estimates. Historical CPU timings do not establish how fast a new GPU run will be.


In [ ]:
# Three batches of each variant on this device, forward and backward, after a warm-up batch.
# RuntimeError/MemoryError during batch processing become table notes. Model construction
# precedes the try block and can still raise; this is not a complete execution preflight.
batches_per_epoch = -(-len(STREAMS.split_rows("train")) // BATCH_SIZE)


def targets_for_loss(batch):
    return batch.labels if STREAMS.task == "classification" else batch.labels.view(-1, 1).float()


def time_variant(variant):
    """Minutes per epoch for one variant, or NaN and the reason it could not be measured."""
    model = blf.build(STREAMS, variant, channels=CHANNELS).to(DEVICE)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)
    criterion = train.loss_function(STREAMS.task)
    batches = iter(blf.loader(STREAMS, "train", batch_size=BATCH_SIZE, shuffle=False))
    try:
        warm_batch = next(batches).to(DEVICE)
        prediction, _ = model(warm_batch)
        criterion(prediction, targets_for_loss(warm_batch)).backward()
        started, measured_batches = time.perf_counter(), 0
        for batch in batches:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            prediction, _ = model(batch)
            criterion(prediction, targets_for_loss(batch)).backward()
            optimizer.step()
            measured_batches += 1
            if measured_batches == 3:
                break
        minutes_per_epoch, timing_note = (
            time.perf_counter() - started
        ) / measured_batches * batches_per_epoch / 60, ""
    except (RuntimeError, MemoryError) as problem:
        minutes_per_epoch, timing_note = (
            float("nan"),
            f"{type(problem).__name__}: {str(problem)[:90]}",
        )
    finally:
        del model, optimizer
        if DEVICE == "mps":
            torch.mps.empty_cache()
    return {
        "params": parameter_count,
        "min/epoch": round(minutes_per_epoch, 2),
        "note": timing_note,
    }


In [5]:
# Each model once, not once per lambda: the sweep reuses the same architecture.
timing = pd.DataFrame(
    [
        {"variant": variant, **time_variant(variant)}
        for variant in dict.fromkeys(variant for variant, _ in RUNS)
    ]
).set_index("variant")
timing["hours @100"] = (timing["min/epoch"] * 100 / 60).round(1)
timing["hours @250"] = (timing["min/epoch"] * 250 / 60).round(1)

runs_per_variant = pd.Series([variant for variant, _ in RUNS]).value_counts()
total_minutes_per_epoch = (timing["min/epoch"] * timing.index.map(runs_per_variant)).sum()
display(timing)
print(
    f"whole sweep, {len(RUNS)} runs: {total_minutes_per_epoch * 100 / 60:.1f} hours at 100 epochs each, "
    f"{total_minutes_per_epoch * 250 / 60:.1f} hours at 250"
)
for variant, timing_note in timing.loc[timing["note"] != "", "note"].items():
    print(f"NOT MEASURED  {variant}: {timing_note}")


,params,min/epoch,note,hours @100,hours @250
variant,,,,,
unimodal:prot,384601,0.16,,0.3,0.7
unimodal:dna,615001,0.07,,0.1,0.3
unimodal:rna,768601,0.07,,0.1,0.3
concat,2751945,0.14,,0.2,0.6
mil,2753246,0.07,,0.1,0.3
cross,7805145,0.51,,0.8,2.1


whole sweep, 9 runs: 2.0 hours at 100 epochs each, 5.1 hours at 250


A finite timing value means the sampled training batches completed. It does not verify a full run or demonstrate that every batch fits in memory. The helper reports selected batch-processing failures, while errors before its `try` block can still propagate. Changing batch size or another training setting would be a separate, documented experiment choice.

### A short run before the long one

The next cell performs a fresh five-epoch protein-only fit whenever executed. It does not use the sweep's saved-result or checkpoint-reuse path. Falling training loss and rising validation score are diagnostics, not requirements that every epoch improve or proof that the reproduction is correct.


In [6]:
# A fresh five-epoch diagnostic; this call does not use the sweep's result-reuse path.
probe = train.fit(
    blf.build(STREAMS, "unimodal:prot", channels=CHANNELS),
    STREAMS,
    train.TrainConfig(
        max_epochs=5,
        batch_size=BATCH_SIZE,
        channels=CHANNELS,
        early_stopping_patience=5,
    ),
    DEVICE,
    log=None,
)
probe_curve = pd.DataFrame(probe.history)[["epoch", "train_loss", "val_score"]]
probe_curve["val_score"] = pd.to_numeric(probe_curve["val_score"], errors="coerce")
display(probe_curve)
falling = probe_curve["train_loss"].iloc[-1] < probe_curve["train_loss"].iloc[0]
rising = probe_curve["val_score"].iloc[-1] > probe_curve["val_score"].iloc[0]
print(
    f"train loss falling: {falling} | validation score rising: {rising} | "
    f"{probe.seconds / 5:.1f} s per epoch"
)


,epoch,train_loss,val_score
0,0,6.247321,0.377599
1,1,5.203534,0.393609
2,2,4.941076,0.418863
3,3,4.739991,0.442122
4,4,4.591699,0.461122


train loss falling: True | validation score rising: True | 4.2 s per epoch


### The sweep

The first cell defines `run_recorded`; the next requests the declared fits in order. Compatible completed results are read back, and compatible interrupted fits can resume from checkpoints. New work receives the shared input/cache identity and execution records. Missing or incompatible evidence can stop a request; resumption is not an unconditional promise.

Checkpoints are normally written every ten epochs, so an interruption may lose work since the last valid checkpoint. Preserve completed artifacts and inspect any reported recovery issue. For an authorized long run, keep the machine powered and prevent sleep.


In [ ]:
# Completed results are preserved. New fits require matching input/cache provenance;
# incomplete runs resume only when checkpoint recovery passes validation.
RUN_CACHE_VERIFICATION = token_embeddings.CacheVerification()


def run_recorded(streams, variant, config, device, results_dir, **kwargs):
    # Historical completed results retain limited legacy checks; new fits record identity.
    records = run_identity.execution_records(
        streams,
        variant,
        config,
        results_dir,
        verification=RUN_CACHE_VERIFICATION,
        repository=CODE_DIR.parent,
        device=device,
    )
    return train.run(streams, variant, config, device, results_dir, **records, **kwargs)


In [7]:
started = time.perf_counter()
for variant, overrides in RUNS:
    config = train.TrainConfig(
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        channels=CHANNELS,
        **overrides,
    )
    run_recorded(STREAMS, variant, config, DEVICE, RESULTS_DIR)
print(f"\nsweep finished in {(time.perf_counter() - started) / 3600:.2f} hours")


  fungal_expression__unimodal-prot__lambda0.0__ch100__appendix__seed0: already run, test 0.7063
  fungal_expression__unimodal-dna__lambda0.0__ch100__appendix__seed0: already run, test 0.8140
  fungal_expression__unimodal-rna__lambda0.0__ch100__appendix__seed0: already run, test 0.7224
  fungal_expression__concat__lambda0.0__ch100__appendix__seed0: already run, test 0.7819
  fungal_expression__mil__lambda0.01__ch100__appendix__seed0: already run, test 0.7869
  fungal_expression__mil__lambda0.5__ch100__appendix__seed0: already run, test 0.7939
  fungal_expression__mil__lambda1.0__ch100__appendix__seed0: already run, test 0.7662
  fungal_expression__mil__lambda0.0__ch100__appendix__seed0: already run, test 0.7854
  fungal_expression__cross__lambda0.0__ch100__appendix__seed0: already run, test 0.8281

sweep finished in 0.00 hours


---

## Did it reproduce?

The next cells distinguish absolute score agreement, single-encoder ordering, fusion-versus-single comparisons, fusion-head ordering and the direction of the entropy ablation. These are descriptive checks of the loaded scores, not a formal acceptance test.

The reader requires the declared benchmark, complete recorded training recipe and selection seed, excludes unrelated records, and withholds duplicate identities. Positive MIL selection requires every declared validation candidate; the zero-entropy ablation stays separate. Coverage is displayed before the scores, and an incomplete comparison is marked unavailable rather than prompting replacement fits. Legacy scores remain readable; configuration agreement does not recover their original inputs or producer.

Verdicts use full-precision values and strict improvement; display rounding does not affect decisions. “Highest observed single-encoder test score” is a descriptive comparison, not a prospective baseline-selection procedure. The later paired analysis retains its fixed DNA baseline. Changing `BENCHMARK` alone does not establish a comparable protocol for another task.


In [ ]:
# Map local variant names to the relevant published reference rows.
PUBLISHED_OF = {
    "unimodal:dna": ("Nucleotide Transformer", TABLE1.loc["Nucleotide Transformer", BENCHMARK]),
    "unimodal:rna": ("RNA-FM", TABLE1.loc["RNA-FM", BENCHMARK]),
    "unimodal:prot": ("ESM-2 (8M)", TABLE1.loc["ESM-2 (8M)", BENCHMARK]),
    "concat": ("Concatenation", TABLE1.loc["Concatenation", BENCHMARK]),
    "mil": ("MIL + Entropy", TABLE1.loc["MIL + Entropy", BENCHMARK]),
    "mil (lam 0)": ("MIL without entropy", TABLE4.loc["MIL without entropy", BENCHMARK]),
    "cross": ("Cross Attention", TABLE1.loc["Cross Attention", BENCHMARK]),
}


In [ ]:
def read_build_results(tconv, seed=SELECTION_SEED, *, mil_lambda=None):
    """Read one complete recipe; unrelated files and duplicates remain diagnostics."""
    config = vars(
        train.TrainConfig(
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            channels=CHANNELS,
            tconv=tconv,
            seed=seed,
        )
    )
    return swap_results.read_results(
        (RESULTS_DIR,),
        benchmark=BENCHMARK,
        config=config,
        encoders=blf.ENCODER_OF_MODALITY,
        variants=("unimodal:dna", "unimodal:rna", "unimodal:prot", "concat", "mil", "cross"),
        mil_lambda=mil_lambda,
        mil_lambdas=(0.0, *LAMBDAS) if mil_lambda is None else (mil_lambda,),
    )


def selected_build_entries(selection):
    """Select positive MIL only when every declared validation candidate is unique."""
    _, selected_lambda = swap_results.validation_choices(
        selection.records,
        singles=(),
        positive_lambdas=LAMBDAS,
    )
    chosen = {}
    for entry in selection.records:
        record = entry["data"]
        name, penalty = record["variant"], record["config"]["entropy_lambda"]
        if name == "mil" and penalty == 0:
            name = "mil (lam 0)"
        elif name == "mil" and penalty != selected_lambda:
            continue
        chosen[name] = entry
    return chosen, selected_lambda


def report_reader_issues(selection):
    if selection.issues:
        print(
            f"{len(selection.issues)} files excluded; inspect this selection's .issues for reasons."
        )
    for duplicate in selection.duplicates:
        print(f"Ambiguous result withheld: {duplicate}")


In [8]:
appendix_selection = read_build_results("appendix")
appendix_entries, chosen_lambda = selected_build_entries(appendix_selection)
report_reader_issues(appendix_selection)

# Keep raw precision for decisions; round only the displayed tables.
comparison_rows = []
for name, entry in appendix_entries.items():
    record = entry["data"]
    paper_name, published_score = PUBLISHED_OF[name]
    comparison_rows.append(
        {
            "variant": name,
            "paper row": paper_name,
            "ours": record["test_score"],
            "published": published_score,
            "difference": record["test_score"] - published_score,
            "epochs": record["epochs"],
            "minutes": record["seconds"] / 60,
        }
    )
comparison = pd.DataFrame(
    comparison_rows,
    columns=["variant", "paper row", "ours", "published", "difference", "epochs", "minutes"],
).set_index("variant")

# Coverage is explicit even when no result is available. Missing work never starts a fit.
comparison_coverage = pd.DataFrame(
    [
        {
            "variant": name,
            "status": "available" if name in appendix_entries else "missing or ambiguous",
        }
        for name in PUBLISHED_OF
    ]
).set_index("variant")
display(comparison_coverage)
print(
    f"positive MIL lambda: {chosen_lambda} (seed {SELECTION_SEED} validation)"
    if chosen_lambda is not None
    else "Positive MIL selection unavailable: the declared validation grid is incomplete."
)
display(comparison.round(3))

finished = pd.DataFrame(
    [
        {
            "variant": entry["data"]["variant"],
            "lambda": entry["data"]["config"]["entropy_lambda"],
            "seed": entry["data"]["config"]["seed"],
            "val": entry["data"]["val_score"],
            "test": entry["data"]["test_score"],
            "epochs": entry["data"]["epochs"],
            "minutes": entry["data"]["seconds"] / 60,
        }
        for entry in appendix_selection.records
    ],
    columns=["variant", "lambda", "seed", "val", "test", "epochs", "minutes"],
)
display(finished.round(3))


gated head: lambda 0.5 chosen on validation from [0.01, 0.5, 1.0]


,paper row,ours,published,difference,epochs,minutes
variant,,,,,,
concat,Concatenation,0.782,0.805,-0.023,33,4.2
cross,Cross Attention,0.828,0.812,0.016,34,18.6
mil (lam 0),MIL without entropy,0.785,0.826,-0.041,35,4.0
mil,MIL + Entropy,0.794,0.824,-0.030,35,4.7
unimodal:dna,Nucleotide Transformer,0.814,0.804,0.010,85,5.0
unimodal:prot,ESM-2 (8M),0.706,0.695,0.011,86,4.8
unimodal:rna,RNA-FM,0.722,0.767,-0.045,27,2.6


,variant,lambda,val,test,epochs,minutes
0,concat,0.00,0.696243,0.781942,33,4.2
1,cross,0.00,0.734546,0.828086,34,18.6
2,mil,0.01,0.701550,0.786872,33,4.6
3,mil,0.00,0.710728,0.785416,35,4.0
4,mil,0.50,0.709015,0.793865,35,4.7
5,mil,1.00,0.662853,0.766228,106,12.1
6,unimodal:dna,0.00,0.703497,0.813991,85,5.0
7,unimodal:prot,0.00,0.663796,0.706285,86,4.8
8,unimodal:rna,0.00,0.638369,0.722432,27,2.6


In [ ]:
def record_comparison_check(check_rows, text, known, passed, detail):
    check_rows.append(
        {
            "claim": text,
            "result": "not yet" if not known else ("ok" if passed else "NO"),
            "detail": detail,
        }
    )


def published_variant_order(rows):
    return sorted(rows, key=lambda k: -TABLE1.loc[rows[k], BENCHMARK])


In [9]:
# Descriptive checks use full precision and complete coverage, not statistical acceptance tests.
# What counts as the published answer is read out of Table 1, so switching BENCHMARK above
# switches the expectations with it.
ours = comparison["ours"].to_dict()
SINGLE_ROWS = {
    "unimodal:dna": "Nucleotide Transformer",
    "unimodal:rna": "RNA-FM",
    "unimodal:prot": "ESM-2 (8M)",
}
FUSION_ROWS = {"concat": "Concatenation", "mil": "MIL + Entropy", "cross": "Cross Attention"}
verdict = []


gaps = comparison["difference"].abs()
complete = all(name in ours for name in PUBLISHED_OF)
record_comparison_check(
    verdict,
    "every variant lands within 0.05 of its published value",
    complete,
    bool((gaps <= 0.05).all()),
    (
        f"largest available gap {gaps.max():.3f} ({gaps.idxmax()})"
        if not gaps.empty
        else "No compatible scores available."
    ),
)

known = all(k in ours for k in SINGLE_ROWS)
published_order = published_variant_order(SINGLE_ROWS)
our_order = sorted((k for k in SINGLE_ROWS if k in ours), key=lambda k: -ours[k])
record_comparison_check(
    verdict,
    "the three single encoders come out in the published order",
    known,
    known
    and len({ours[k] for k in SINGLE_ROWS}) == len(SINGLE_ROWS)
    and our_order == published_order,
    "published "
    + " > ".join(published_order)
    + "; ours "
    + " > ".join(f"{k} {ours[k]:.3f}" for k in our_order),
)

fusions = [k for k in FUSION_ROWS if k in ours]
singles = [k for k in SINGLE_ROWS if k in ours]
known = len(fusions) == 3 and len(singles) == 3
best_single = max((ours[k] for k in singles), default=float("nan"))
record_comparison_check(
    verdict,
    "every fusion head exceeds the highest observed single-encoder test score",
    known,
    known and all(ours[f] > best_single for f in fusions),
    f"highest available single test score {best_single:.3f}; "
    + ", ".join(f"{f} {ours[f]:.3f}" for f in fusions),
)

published_best = published_variant_order(FUSION_ROWS)[0]
known = len(fusions) == 3
record_comparison_check(
    verdict,
    f"the best fusion head is the paper's best here ({published_best})",
    known,
    known and all(ours[published_best] > ours[k] for k in fusions if k != published_best),
    ", ".join(f"{f} {ours[f]:.3f}" for f in fusions),
)

# Missing ablations are reported as unavailable; a tie is distinct from a decrease.
known = "mil" in ours and "mil (lam 0)" in ours
if known:
    our_gain = ours["mil"] - ours["mil (lam 0)"]
    paper_gain = (
        TABLE1.loc["MIL + Entropy", BENCHMARK] - TABLE4.loc["MIL without entropy", BENCHMARK]
    )
    record_comparison_check(
        verdict,
        "entropy moves the gated head the way the paper's own ablation does",
        True,
        np.sign(our_gain) == np.sign(paper_gain),
        f"ours {our_gain:+.3f}, paper {paper_gain:+.3f}; note the paper's text says entropy "
        f"improves performance while its Table 4 has it {paper_gain:+.3f} on this benchmark",
    )

else:
    record_comparison_check(
        verdict,
        "entropy moves the gated head the way the paper's own ablation does",
        False,
        False,
        "Both selected positive MIL and zero-entropy MIL are required.",
    )

pd.DataFrame(verdict).set_index("claim")


,result,detail
claim,,
every variant lands within 0.05 of its published value,ok,largest gap 0.045 (unimodal:rna)
the three single encoders come out in the published order,ok,published unimodal:dna > unimodal:rna > unimod...
every fusion head beats the best single encoder,NO,"best single 0.814; concat 0.782, mil 0.794, cr..."
the best fusion head is the paper's best here (mil),NO,"concat 0.782, mil 0.794, cross 0.828"
entropy moves the gated head the way the paper's own ablation does,NO,"ours +0.009, paper -0.002; note the paper's te..."


---

## The transposed convolution: the paper's two settings are not equivalent

The first sweep uses the appendix setting. To understand its effect before another fit, the next cell sets convolution weights to one and the bias to zero, then sends an impulse at selected DNA-token positions.

For this example, 120 nucleotides form exactly 20 six-mers and 40 codons. Kernel 2, stride 2 and padding 0 produce 40 positions. Kernel 3, stride 2 and padding 2 produce 37 before adjustment to the frame. The printed nonzero positions reveal different coverage and boundary behavior; this example deliberately contains no remainder tokens.

This checks a specific mapping operation. It does not establish how trained convolution weights encode biological information or whether equal output lengths imply equal correspondence.


In [10]:
# Where does one DNA 6-mer token land on the codon frame? With the convolution set to copy
# its input, token t should light up codons 2t and 2t+1, because a 6-mer covers two codons.
for setting in codon_grid.TCONV_SETTINGS:
    probe_grid = codon_grid.CodonGrid(1, tconv=setting).eval()
    with torch.no_grad():
        probe_grid.dna_upsample.weight.fill_(1.0)
        probe_grid.dna_upsample.bias.fill_(0.0)
    dna_token_count, codon_count = 20, 40  # 120 nt: 20 six-mers, 40 codons
    produced = (
        (dna_token_count - 1) * probe_grid.stride - 2 * probe_grid.padding + probe_grid.kernel
    )
    print(
        f"{setting:<9} kernel {probe_grid.kernel} stride {probe_grid.stride} "
        f"padding {probe_grid.padding} -> {produced} positions for a {codon_count}-codon frame"
    )
    for token_index in (0, 1, 5, 19):
        impulse = torch.zeros(1, dna_token_count, 1)
        impulse[0, token_index, 0] = 1.0
        with torch.no_grad():
            landed, _ = probe_grid(
                impulse,
                torch.zeros(1, 120, 1),
                torch.zeros(1, codon_count, 1),
                torch.tensor([codon_count]),
                torch.tensor([dna_token_count]),
            )
        observed_codons = (landed["dna"][0, :, 0] != 0).nonzero().flatten().tolist()
        expected_codons = [2 * token_index, 2 * token_index + 1]
        print(
            f"    6-mer {token_index:>2} -> codons {str(observed_codons):<14}"
            + ("" if observed_codons == expected_codons else f"should be {expected_codons}")
        )
    print()


appendix  kernel 3 stride 2 padding 2 -> 37 positions for a 40-codon frame
    6-mer  0 -> codons [0]           should be [0, 1]
    6-mer  1 -> codons [0, 1, 2]     should be [2, 3]
    6-mer  5 -> codons [8, 9, 10]    should be [10, 11]
    6-mer 19 -> codons [36]          should be [38, 39]

equation  kernel 2 stride 2 padding 0 -> 40 positions for a 40-codon frame
    6-mer  0 -> codons [0, 1]        
    6-mer  1 -> codons [2, 3]        
    6-mer  5 -> codons [10, 11]      
    6-mer 19 -> codons [38, 39]      



### The hypothesis and the paired configuration change

Changing DNA placement could affect position-wise concatenation or gated fusion differently from attention over a joint context. The following comparison therefore repeats the grid-using variants under Equation 1 while retaining the other requested settings. Native single-encoder baselines bypass the grid and are not requested again in this section.

The [recorded convolution comparison](../../docs/run-records.md#2026-10-17-stage3-tconv-rerun-the-papers-two-transposed-convolution-settings-compared) found increases for concatenation and gated variants, with a small decrease for cross-attention. That supports investigating the placement choice, but one configuration comparison does not isolate a biological mechanism; the positive MIL weight is selected separately for each setting.

The earlier explanation treated a DNA gate weight near zero as evidence of damaged input. The same run record explicitly corrects that reading: concentrated gates persisted after the convolution change and were associated with the entropy penalty. Gate weights describe the fitted computation; they do not establish that a modality is damaged, irrelevant or biologically unimportant.

The score reader and gate-weight display use the same selected seed-0 records. Missing candidates remain visible. Gate summaries identify their seed and source file.

**Historical snapshot.** The retained table belongs to the original convolution comparison. Later prediction reruns replaced some selected Equation 1 seed-0 files; for example, the retained concatenation score is about 0.797, while the replacement file stores about 0.793. Reading the current files therefore does not reconstruct that original controlled comparison. Keep its saved output and run record distinct from any later refresh.


In [11]:
# Repeat only grid-using variants under Equation 1, in the original order.
EQUATION_RUNS = (
    [("concat", {})]
    + [("mil", {"entropy_lambda": lam}) for lam in LAMBDAS]
    + [("mil", {"entropy_lambda": 0.0}), ("cross", {})]
)
started = time.perf_counter()
for variant, overrides in EQUATION_RUNS:
    run_recorded(
        STREAMS,
        variant,
        train.TrainConfig(
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            channels=CHANNELS,
            tconv="equation",
            **overrides,
        ),
        DEVICE,
        RESULTS_DIR,
    )
print(f"\nrerun finished in {(time.perf_counter() - started) / 3600:.2f} hours")


  fungal_expression__concat__lambda0.0__ch100__equation__seed0: already run, test 0.7968
  fungal_expression__mil__lambda0.01__ch100__equation__seed0: already run, test 0.8018
  fungal_expression__mil__lambda0.5__ch100__equation__seed0: already run, test 0.7943
  fungal_expression__mil__lambda1.0__ch100__equation__seed0: already run, test 0.7708
  fungal_expression__mil__lambda0.0__ch100__equation__seed0: already run, test 0.8022
  fungal_expression__cross__lambda0.0__ch100__equation__seed0: already run, test 0.8235

rerun finished in 0.00 hours


In [ ]:
def scores_for_convolution(tconv):
    """Use exactly the same seed/recipe selection for scores and modality weights."""
    selection = read_build_results(tconv)
    entries, selected_lambda = selected_build_entries(selection)
    scores = {name: entry["data"]["test_score"] for name, entry in entries.items()}
    return scores, selected_lambda, entries, selection


In [12]:
appendix, lambda_appendix, appendix_entries, appendix_selection = scores_for_convolution("appendix")
equation, lambda_equation, equation_entries, equation_selection = scores_for_convolution("equation")
report_reader_issues(appendix_selection)
report_reader_issues(equation_selection)

# Keep missing configurations visible instead of silently dropping their rows.
grid_users = ("concat", "mil", "mil (lam 0)", "cross")
settings = pd.DataFrame(
    {
        "published": {name: PUBLISHED_OF[name][1] for name in grid_users},
        "appendix (k3 s2 p2)": {name: appendix.get(name) for name in grid_users},
        "Equation 1 (k2 s2)": {name: equation.get(name) for name in grid_users},
    }
)
settings["change"] = settings["Equation 1 (k2 s2)"] - settings["appendix (k3 s2 p2)"]
settings["Eq 1 vs published"] = settings["Equation 1 (k2 s2)"] - settings["published"]
print("Current selected files; these do not reconstruct the original convolution-run snapshot.")
display(settings.round(3))
print(f"lambda chosen on validation: appendix {lambda_appendix}, Equation 1 {lambda_equation}")

# Weights now come from the exact MIL records supplying the score table.
for tconv, entries in (("appendix", appendix_entries), ("equation", equation_entries)):
    if "mil" not in entries:
        continue
    entry = entries["mil"]
    record = entry["data"]
    if not record.get("alpha"):
        continue
    print(
        f"{tconv} seed {record['config']['seed']} ({entry['path'].name}): "
        + "  ".join(f"{modality} {weight:.2f}" for modality, weight in record["alpha"].items())
    )


,published,appendix (k3 s2 p2),Equation 1 (k2 s2),change,Eq 1 vs published
concat,0.805,0.782,0.797,0.015,-0.008
mil,0.824,0.794,0.802,0.008,-0.022
mil (lam 0),0.826,0.785,0.802,0.017,-0.024
cross,0.812,0.828,0.824,-0.005,0.012


lambda chosen on validation: appendix 0.5, Equation 1 0.01
appendix  gated attention weights: dna 0.00  rna 0.48  prot 0.52
equation  gated attention weights: dna 0.20  rna 0.20  prot 0.60


---

## How much of this is measurement noise?

The fungal test split contains 430 rows. Before comparing two models, the next cell illustrates uncertainty for individual Spearman scores using the existing z-scale approximation, with standard error `1.03 / sqrt(n - 3)`. Its intervals are approximate descriptions of separate scores, not intervals for model differences.

The published fungal MIL score of 0.824 and the original NT baseline of 0.804 differ by 0.020. Whether a local difference is resolved cannot be decided by comparing that gap with either score's marginal interval: models evaluated on the same rows have paired errors. That motivates the prediction-level comparison below.


In [13]:
# Approximate marginal intervals for individual Spearman scores, not paired differences.
# Retain the existing z-scale approximation: 1.03 / sqrt(n - 3).
test_rows = len(STREAMS.split_rows("test"))
z_error = 1.03 / np.sqrt(test_rows - 3)
interval = pd.DataFrame(
    [
        {
            "score": s,
            "95% low": np.tanh(np.arctanh(s) - 1.96 * z_error),
            "95% high": np.tanh(np.arctanh(s) + 1.96 * z_error),
        }
        for s in sorted(
            set(
                list(settings["published"])
                + list(settings["Equation 1 (k2 s2)"].dropna())
                + [appendix[k] for k in appendix if k.startswith("unimodal")]
            )
        )
    ]
)
interval["half-width"] = (interval["95% high"] - interval["95% low"]) / 2
print(f"test split: {test_rows} rows")
display(interval.round(3).set_index("score"))


test split: 430 rows


,95% low,95% high,half-width
score,,,
0.706,0.654,0.752,0.049
0.722,0.672,0.766,0.047
0.797,0.758,0.830,0.036
0.802,0.764,0.834,0.035
0.802,0.765,0.834,0.035
0.805,0.768,0.837,0.034
0.812,0.776,0.843,0.033
0.814,0.778,0.844,0.033
0.824,0.789,0.853,0.032


### Test-row uncertainty and training-seed variation

A paired bootstrap resamples the same test-row positions for both models and recalculates their score difference. It needs per-row targets and predictions. The present shared trainer saves those for new fits; `require_predictions=True` refuses a completed result that lacks them instead of retraining it. Retained logs showing historical replacement fits describe the earlier execution, not current reuse behavior.

Training seeds address a different question: how much the fitted result changes with initialization, shuffling and other seeded operations. The declared repeats use seeds 0, 1 and 2, hold the recorded Equation 1 MIL weight at `CLAIM_LAMBDA=0.01`, and compare three fusion heads with a **fixed DNA baseline**. DNA was consistent with the historical native validation ranking, but this cell hard-codes it; it does not implement a prospective validation-only baseline selection.

The bootstrap averages the per-seed scores within each draw; it does not average predictions into an ensemble or eliminate training uncertainty. Each model must have every requested seed, valid arrays and matching ordered targets. Recorded input signatures must agree for modern fits. The old fungal fits instead require exact path and hash matches to the existing verification report; this qualifies later comparison evidence without reconstructing their producer or historical input identity. Modern and historical evidence are not silently mixed.

Before any seed summary is displayed, all requested seeds must share matching recorded input evidence or belong to the same qualified historical verification. Missing, conflicting or unsupported evidence leaves the aggregate columns blank with an explicit summary status. Recorded score summaries do not require prediction arrays; pairing adds that requirement and checks the arrays against their recorded targets.

A missing or incompatible model withholds only contrasts that need it, and an undefined observed or resampled score withholds the interval. The statistic remains 4,000 paired row resamples with seed 0 and unadjusted percentile intervals; seed variability and biological relatedness remain separate limitations.

**Saved-output distinction.** The following analysis cell retains a per-seed interval table from an older execution, while its current source specifies seed-averaged contrasts. The [fungal saved-result verification](../../docs/run-records.md#fungal-equation-1-saved-result-verification) independently evaluated the aggregate procedure and records its results and limitations. The source, retained output and verification record serve different roles; the saved outputs and verification report remain unchanged by the reader repairs.


In [14]:
# A fixed DNA baseline and three fusion heads, under the existing Equation 1 recipe.
# Completed fits lacking predictions are preserved; require_predictions refuses replacement.
CLAIM_RUNS = [
    ("unimodal:dna", {}),
    ("concat", {}),
    ("mil", {"entropy_lambda": CLAIM_LAMBDA}),
    ("cross", {}),
]
started = time.perf_counter()
for seed in SEEDS:
    print(f"\n--- seed {seed} ---")
    for variant, overrides in CLAIM_RUNS:
        run_recorded(
            STREAMS,
            variant,
            train.TrainConfig(
                batch_size=BATCH_SIZE,
                max_epochs=MAX_EPOCHS,
                channels=CHANNELS,
                tconv="equation",
                seed=seed,
                **overrides,
            ),
            DEVICE,
            RESULTS_DIR,
            require_predictions=True,
        )
print(f"\nseed sweep finished in {(time.perf_counter() - started) / 3600:.2f} hours")



--- seed 0 ---
  fungal_expression__unimodal-dna__lambda0.0__ch100__equation__seed0: training
    epoch   0  train loss 5.4896  val 0.5162  best 0.5162 @ 0
    epoch  10  train loss 3.3860  val 0.6486  best 0.6486 @ 10
    epoch  20  train loss 2.4631  val 0.6702  best 0.6702 @ 20
    epoch  30  train loss 1.7329  val 0.6829  best 0.6829 @ 30
    epoch  40  train loss 1.0838  val 0.6916  best 0.6916 @ 40
    epoch  50  train loss 0.6398  val 0.6971  best 0.6971 @ 50
    epoch  60  train loss 0.3452  val 0.6988  best 0.6988 @ 60
    epoch  70  train loss 0.1991  val 0.6989  best 0.6994 @ 64
    epoch  80  train loss 0.1990  val 0.6993  best 0.6994 @ 64
  fungal_expression__unimodal-dna__lambda0.0__ch100__equation__seed0: test 0.8125 (val 0.6994 at epoch 64, 4.1 min)
  fungal_expression__concat__lambda0.0__ch100__equation__seed0: retraining, the stored result has no per-row predictions
  fungal_expression__concat__lambda0.0__ch100__equation__seed0: training
    epoch   0  train loss 5.5

In [ ]:
def claim_input_evidence(entry):
    """Identify a score cohort using recorded inputs or the exact qualified historical files."""
    path, record = entry["path"], entry["data"]
    payload = path.read_bytes()
    if json.loads(payload) != record:
        raise ValueError("Result changed after it was read; reload the inventory.")

    if "identity" in record:
        # Invalid modern identities must not fall back to a historical exception.
        inputs = run_identity.recorded_blf_inputs(record)
        return {"kind": "recorded inputs", "inputs": inputs}

    audit_bytes = FUNGAL_VERIFICATION.read_bytes()
    if hashlib.sha256(audit_bytes).hexdigest() != FUNGAL_VERIFICATION_SHA256:
        raise ValueError("Historical verification report differs from the selected report.")
    audit = json.loads(audit_bytes)
    if audit.get("status") != "passed_internal_consistency_only":
        raise ValueError("Historical verification did not pass.")
    matches = [
        item
        for item in audit["inputs"]
        if (CODE_DIR.parent / item["path"]).resolve() == path.resolve()
    ]
    if len(matches) != 1:
        raise ValueError("No unique historical verification entry for this result.")
    match = matches[0]
    if (
        match.get("status") != "passed"
        or match.get("sha256") != hashlib.sha256(payload).hexdigest()
        or match.get("variant") != record["variant"]
        or match.get("config") != record["config"]
        or match.get("rows") != len(record["test_truth"])
    ):
        raise ValueError("Result no longer matches the historical verification entry.")
    # This identifies accepted later evidence, never an inferred producer identity.
    return {"kind": "qualified historical files", "report": FUNGAL_VERIFICATION_SHA256}


def paired_input_evidence(entry):
    """Pairing additionally verifies the actual saved targets against their recorded identity."""
    evidence = claim_input_evidence(entry)
    if evidence["kind"] == "recorded inputs":
        target_record = run_identity.array_record(entry["data"]["test_truth"], np.float32)
        if target_record != evidence["inputs"]["split_targets"]["test"]:
            raise ValueError("Saved targets do not match their recorded fingerprint.")
    return evidence


In [ ]:
def read_claim_results():
    """Collect requested seeds without silently substituting a different cohort."""
    if not SEEDS or len(set(SEEDS)) != len(SEEDS):
        raise ValueError("SEEDS must contain distinct requested seed IDs.")
    results = {variant: {} for variant, _ in CLAIM_RUNS}
    issues = []
    for seed in SEEDS:
        selection = read_build_results("equation", seed, mil_lambda=CLAIM_LAMBDA)
        issues.extend(selection.issues)
        issues.extend({"source": "", "reason": item} for item in selection.duplicates)
        for entry in selection.records:
            name = entry["data"]["variant"]
            if name in results:
                results[name][seed] = entry
    return results, issues


def summarize_claim_results(results):
    """Require complete seeds and matching cohort evidence before showing aggregates."""
    summaries = []
    for variant, entries in results.items():
        available = tuple(seed for seed in SEEDS if seed in entries)
        missing = tuple(seed for seed in SEEDS if seed not in entries)
        issue, evidence = "", None
        if missing:
            issue = f"missing or ambiguous seeds: {missing}"
        else:
            for seed in SEEDS:
                entry = entries[seed]
                if (
                    entry["data"]["variant"] != variant
                    or entry["data"]["config"]["seed"] != seed
                ):
                    issue = "record variant/seed differs from its requested identity"
                    break
                try:
                    signature = claim_input_evidence(entry)
                except (OSError, ValueError, KeyError, TypeError) as error:
                    issue = str(error)
                    break
                if evidence is not None and signature != evidence:
                    issue = "input evidence differs across seeds"
                    break
                evidence = signature
        ready = not issue
        scores = [entries[seed]["data"]["test_score"] for seed in available]
        summaries.append(
            {
                "variant": variant,
                "seeds": len(available),
                "available seeds": available,
                "missing seeds": missing,
                "summary status": issue or "ready",
                "input evidence": evidence["kind"] if ready and evidence else "",
                "mean": np.mean(scores) if ready else np.nan,
                "min": min(scores) if ready else np.nan,
                "max": max(scores) if ready else np.nan,
                "spread": max(scores) - min(scores) if ready else np.nan,
                "published": PUBLISHED_OF[variant][1],
            }
        )
    summary = pd.DataFrame(summaries).set_index("variant")
    summary["mean vs published"] = summary["mean"] - summary["published"]
    return summary


In [ ]:
claim_results, claim_reader_issues = read_claim_results()
spread = summarize_claim_results(claim_results)
if claim_reader_issues:
    print(f"{len(claim_reader_issues)} exclusions/ambiguities; details: claim_reader_issues.")


In [ ]:
def prepare_paired_results(results):
    """Validate every seed before a model can enter any paired contrast."""
    ready, coverage = {}, []
    for variant, entries in results.items():
        missing = tuple(seed for seed in SEEDS if seed not in entries)
        issue, evidence, truth, predictions = "", None, None, []
        if missing:
            issue = f"missing or ambiguous seeds: {missing}"
        else:
            for seed in SEEDS:
                entry = entries[seed]
                record = entry["data"]
                issue = swap_results.prediction_issue(record, truth, task="regression")
                if issue:
                    break
                if record["variant"] != variant or record["config"]["seed"] != seed:
                    issue = "record variant/seed differs from its requested identity"
                    break
                try:
                    signature = paired_input_evidence(entry)
                except (OSError, ValueError, KeyError, TypeError) as error:
                    issue = str(error)
                    break
                if evidence is not None and signature != evidence:
                    issue = "input evidence differs across seeds"
                    break
                evidence = signature
                truth = record["test_truth"]
                predictions.append(np.asarray(record["test_prediction"]))
        if not issue:
            ready[variant] = {
                "truth": np.asarray(truth),
                "predictions": predictions,
                "evidence": evidence,
            }
        coverage.append(
            {
                "variant": variant,
                "requested seeds": tuple(SEEDS),
                "available seeds": tuple(seed for seed in SEEDS if seed in entries),
                "paired status": issue or "ready",
                "input evidence": evidence["kind"] if evidence and not issue else "",
            }
        )
    return ready, coverage


def paired_contrast_issue(ready, first, second):
    """Pair only complete models sharing ordered targets and the same input evidence."""
    if first not in ready or second not in ready:
        return "one or both models lack complete, valid requested seeds"
    left, right = ready[first], ready[second]
    if not np.array_equal(left["truth"], right["truth"]):
        return "ordered targets differ between models"
    if left["evidence"] != right["evidence"]:
        return "input evidence differs between models"
    return None


In [ ]:
def bootstrap_contrast(ready, first, second, *, draws=4000, seed=0):
    """Keep the historical mean-of-seed-scores statistic and paired row resampling."""
    issue = paired_contrast_issue(ready, first, second)
    if issue:
        raise ValueError(issue)
    truth = ready[first]["truth"]
    rows = np.arange(len(truth))

    def averaged(variant, pick):
        scores = [
            train.score_rows("regression", truth[pick], prediction[pick])
            for prediction in ready[variant]["predictions"]
        ]
        if not np.isfinite(scores).all():
            raise ValueError("Undefined observed or resampled score; interval withheld.")
        return np.mean(scores)

    observed = averaged(first, rows) - averaged(second, rows)
    rng = np.random.default_rng(seed)
    differences = []
    for _ in range(draws):
        pick = rng.integers(0, len(truth), len(truth))
        differences.append(averaged(first, pick) - averaged(second, pick))
    low, high = np.percentile(differences, [2.5, 97.5])
    return {
        "difference": observed,
        "95% low": low,
        "95% high": high,
        "P(>0)": (np.asarray(differences) > 0).mean(),
        "resolved": "yes" if low > 0 or high < 0 else "no",
    }


In [15]:
display(spread.round(3))
paired_models, paired_coverage = prepare_paired_results(claim_results)
display(pd.DataFrame(paired_coverage).set_index("variant"))
comparisons = [
    ("concat minus fixed DNA baseline", "concat", "unimodal:dna"),
    ("mil minus fixed DNA baseline", "mil", "unimodal:dna"),
    ("cross minus fixed DNA baseline", "cross", "unimodal:dna"),
    ("cross minus mil", "cross", "mil"),
    ("cross minus concat", "cross", "concat"),
    ("mil minus concat", "mil", "concat"),
]
interval_rows = []
for label, first, second in comparisons:
    issue = paired_contrast_issue(paired_models, first, second)
    if issue:
        interval_rows.append({"comparison": label, "status": issue})
        continue
    try:
        interval = bootstrap_contrast(paired_models, first, second)
    except ValueError as error:
        interval_rows.append({"comparison": label, "status": str(error)})
        continue
    interval_rows.append({"comparison": label, "status": "available", **interval})
display(pd.DataFrame(interval_rows).set_index("comparison").round(3))
print(
    "Intervals condition on this split and these fitted seeds; they are unadjusted for\n"
    "multiple comparisons. P(>0) is the fraction of positive resamples, not a p-value.\n"
    "Qualified historical files retain unknown producer/input provenance."
)


,seeds,mean,min,max,spread,published,mean vs published
variant,,,,,,,
unimodal:dna,3,0.807,0.801,0.813,0.011,0.804,0.003
concat,3,0.791,0.789,0.793,0.004,0.805,-0.014
mil,3,0.801,0.797,0.804,0.007,0.824,-0.023
cross,3,0.823,0.814,0.831,0.017,0.812,0.011


minus best single  95% low  95% high  P(fusion wins)
fusion head seed                                                      
concat      0                -0.020   -0.054     0.015           0.136
            1                -0.008   -0.043     0.027           0.336
            2                -0.017   -0.051     0.016           0.158
mil         0                -0.011   -0.049     0.028           0.298
            1                 0.003   -0.034     0.043           0.587
            2                -0.009   -0.049     0.030           0.338
cross       0                 0.011   -0.024     0.047           0.732
            1                 0.013   -0.018     0.048           0.818
            2                 0.025   -0.010     0.060           0.930

A head beats the best single encoder only where the interval excludes zero.


## Deviations recorded for this step

These choices accompany the inferred length filter and grid-preserving protein translation from the earlier notebooks.

| Choice | Implemented treatment | Interpretation limit |
| --- | --- | --- |
| TextCNN width | 100 channels for the main variants; optional 1,280-channel MIL fit | Controls one capacity difference, without making all architectures equivalent |
| DNA upsampling | Appendix first, then Equation 1 with separate results | Specifications differ; output shape does not resolve remainder-token coverage |
| DNA projection | 512-wide hidden states projected to the named width of 600 | Retains the operation despite the paper's vocabulary/hidden-width ambiguity |
| Row-specific DNA masking | Trim each row to its own convolution output before frame adjustment | Checks padding behavior, not biological correspondence |
| Gated summaries | Mean over real codon positions | Avoids including artificial padding in modality summaries |
| Stopping and scheduler monitor | Validation score; restore the best validation state | A different monitor could select different weights or stopping time |
| Undefined validation score | Count as no improvement under the shared trainer's best-state safeguards | Does not make an undefined metric informative |

Ab1 remains unavailable in the inspected release, so its task-specific comparison is omitted. The table of published values does not establish how an unperformed local Ab1 experiment would behave. The [run records](../../docs/run-records.md#fungal-equation-1-saved-result-verification) retain additional limitations concerning historical selection, producer identity and paired evaluation.


## The optional extra run: does head width explain any gap?

`RUN_WIDE_HEAD` in Setup controls a separate MIL fit using 1,280 TextCNN channels, Equation 1 and the selected positive entropy weight. It examines whether the narrower shared head contributes to the discrepancy with the paper's gated result.

This changes head capacity and resource requirements together. Its runtime is not established by the three-batch timing of the 100-channel models, and a better score would not by itself isolate a fusion-method effect. The flag remains off by default.


In [16]:
# RUN_WIDE_HEAD is configured in Setup. Keep the existing optional fit and reuse checks.

if RUN_WIDE_HEAD:
    wide = run_recorded(
        STREAMS,
        "mil",
        train.TrainConfig(
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            channels=1280,
            entropy_lambda=lambda_equation,
            tconv="equation",
        ),
        DEVICE,
        RESULTS_DIR,
        require_predictions=True,
    )
    print(
        f"gated head at 1,280 channels: {wide.test_score:.3f} "
        f"(at 100 channels: {equation['mil']:.3f}, published: {PUBLISHED_OF['mil'][1]})"
    )
else:
    print("not run; set RUN_WIDE_HEAD = True above to run it")


not run; set RUN_WIDE_HEAD = True above to run it


---

## How the reference supports further experiments

This notebook supplies a shared implementation, an explicit appendix-versus-Equation-1 comparison and a fungal reference with documented limitations. The convolution comparison motivated the Equation 1 configuration used in the [reference benchmark comparisons](blf_reference_benchmarks.ipynb); both settings and their completed results remain available.

The [verified fungal summaries](../../docs/run-records.md#fungal-equation-1-saved-result-verification) leave the fusion-versus-fixed-DNA gains unresolved under their paired intervals. Cross-attention exceeds the other two fusion heads under that qualified analysis. Neither result establishes a biological mechanism or proves that fusion cannot help.

The [reference benchmark notebook](blf_reference_benchmarks.ipynb) extends this build across the available datasets. The [Stage 4 guide](../stage4/README.md) connects those references to encoder replacements and [proposed controlled tests](../stage4/README.md#proposed-tests-of-fusion-explanations) of their explanations. Additional split designs, attention variants and conditional Mamba work remain separate experiments, not prerequisites silently completed by this build check.
